# 📊 EDA — Exploratory Data Analysis

**Tech Challenge Fase 1 · Churn Predictor**

Este notebook explora o dataset Telco Customer Churn (IBM) com foco em:
1. **Volume e qualidade** dos dados.
2. **Distribuições** de features e target.
3. **Relações** entre features e churn.
4. **Data readiness** — o dataset está pronto para modelagem?

In [ ]:
import sys
from pathlib import Path

# Adiciona src ao path se rodando standalone
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from churn_predictor.data.loader import clean_data, load_raw_data

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

## 1. Carregamento do dataset

In [ ]:
df = load_raw_data()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

## 2. Qualidade dos dados

### Missing values

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
missing_df[missing_df["missing"] > 0]

**Observação:** `TotalCharges` tem alguns valores faltantes — investigar.

In [ ]:
# Quem são os clientes com TotalCharges nulo?
df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "Churn"]]

✅ **Insight:** clientes com `TotalCharges` nulo têm `tenure=0` — são clientes recém-adquiridos, ainda não cobrados. Imputar com 0 é semanticamente correto.

### Duplicatas

In [ ]:
print(f"Duplicatas por customerID: {df['customerID'].duplicated().sum()}")
print(f"Duplicatas (linhas inteiras): {df.duplicated().sum()}")

## 3. Distribuição da target

In [ ]:
churn_dist = df["Churn"].value_counts(normalize=True) * 100
print(churn_dist)

fig, ax = plt.subplots(figsize=(6, 4))
churn_dist.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
ax.set_title("Distribuição da target (Churn)")
ax.set_ylabel("% de clientes")
ax.set_xlabel("Churn")
for i, v in enumerate(churn_dist.values):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

✅ **Insight:** Dataset desbalanceado — ~26% de churn. Vamos precisar lidar com isso via `pos_weight` na MLP e `class_weight='balanced'` nos baselines. PR-AUC será métrica primária.

## 4. Análise de features numéricas

In [ ]:
df_clean = clean_data(df)

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df_clean[numeric_cols].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df_clean, x=col, hue="Churn", kde=True, ax=ax, palette=["#2ecc71", "#e74c3c"])
    ax.set_title(f"Distribuição de {col} por Churn")
plt.tight_layout()
plt.show()

✅ **Insights:**
- **`tenure`**: clientes que cancelam têm tenure baixo (concentração em 0-12 meses). Quem passa de 1 ano tende a ficar.
- **`MonthlyCharges`**: distribuição bimodal. Churners pagam mais em média (planos premium = expectativa maior = mais sensíveis).
- **`TotalCharges`**: correlacionado com tenure (lógico). Churners têm TotalCharges baixo.

## 5. Análise de features categóricas

In [ ]:
categorical_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "Contract", "PaperlessBilling", "PaymentMethod", "InternetService",
]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for ax, col in zip(axes.flatten(), categorical_cols):
    crosstab = pd.crosstab(df_clean[col], df_clean["Churn"], normalize="index") * 100
    crosstab.plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#e74c3c"])
    ax.set_title(f"Churn rate por {col}")
    ax.set_ylabel("%")
    ax.legend(["Não Churn", "Churn"], loc="upper right")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

✅ **Insights principais:**
- **`Contract`**: contratos mensais têm churn de ~43%, anuais ~11%, bianuais ~3%. **Maior preditor**.
- **`InternetService`**: fiber optic tem churn quase 2× maior que DSL.
- **`PaymentMethod`**: electronic check tem churn ~45%, vs. ~15% para débito automático.
- **`SeniorCitizen`**: idosos têm churn maior (~42% vs. ~24%).
- **`Partner`/`Dependents`**: clientes solteiros sem dependentes têm churn maior.

## 6. Correlações

In [ ]:
df_corr = df_clean.copy()
# Encode rápido para correlação
for col in df_corr.select_dtypes(include=["object"]).columns:
    df_corr[col] = pd.Categorical(df_corr[col]).codes

corr_with_target = df_corr.corr()["Churn"].drop("Churn").sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.set_title("Correlação com Churn (ordenada por |corr|)")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 7. Análise de tenure por contrato

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df_clean, x="Contract", y="tenure", hue="Churn", split=True, ax=ax, palette=["#2ecc71", "#e74c3c"])
ax.set_title("Distribuição de tenure por tipo de contrato e churn")
plt.tight_layout()
plt.show()

## 8. Data Readiness — Resumo

| Aspecto | Status | Notas |
|---------|--------|-------|
| Volume | ✅ OK | 7.043 registros, suficiente para o problema |
| Missing values | ✅ Resolvido | TotalCharges imputado com 0 (clientes novos) |
| Duplicatas | ✅ OK | Sem duplicatas |
| Tipos | ✅ OK | Após `pd.to_numeric(TotalCharges)` |
| Distribuição da target | ⚠️ Desbalanceado (~26%) | Tratar com `pos_weight` |
| Outliers numéricos | ✅ OK | Sem outliers extremos em tenure/charges |
| Cardinalidade categórica | ✅ Baixa | Max 4 categorias por feature |
| Correlação | ✅ Sinais claros | Contract, tenure, InternetService são top preditores |

✅ **Pronto para modelagem.** Próximos notebooks: `02_baselines.ipynb` e `03_mlp.ipynb`.